# Модуль 13. Разбор кейсов: Шаблонный ретросинтез и выбор маршрута

**Модуль 13. Автор/аффилиация:** автор курса уточняет перед публикацией. **Среда:** WSL/uv/Python 3.12. **Данные:** Цель — метилацетат COC(C)=O; routes.csv содержит два вымышленных линейных маршрута.

Выполнить прозрачное структурное разбиение и проверить упрощённую экономическую модель.

**Workflow:** Эфир → реакционный SMARTS → санитарная проверка продуктов → выходы → сравнение допущений.

**Результаты обучения:** объяснить вход/политику; воспроизвести расчёт и отрицательный контроль; ограничить научный вывод.

Оцените результат до запуска. После разбора переходите во [второй notebook](lab13_practice.ipynb). [Установка среды](../SETUP.md).

[Связанная глава](../../Module_theory/Module%2013_retrosynthesis.md).

## 0. Подготовка и паспорт запуска

Запускайте ячейки сверху вниз после Restart Kernel. Путь Python должен указывать на проектную `.venv`; устанавливать пакеты в ячейке не нужно. Эта ячейка только определяет пути и создаёт отдельный каталог результатов разбора.

In [1]:
from pathlib import Path
import sys, json, math, csv, time, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'Labs/chemistry.py').is_file())
for folder in [ROOT/'Stepik_course/practice', ROOT/'Labs']:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
DATA = ROOT/'Labs/datasets'
OUT = ROOT/'Labs/results/lab13/worked'
OUT.mkdir(parents=True, exist_ok=True)
started = time.perf_counter()
print('Python:', sys.executable)


Python: /home/miakhlov/course-notebooks/.venv/bin/python


## 1. Ретросинтетический шаблон как гипотеза

У заданного эфира разрываем одну связь по реакционному SMARTS. Atom maps связывают атомы исходника с продуктами, но правило не выбирает условия и не гарантирует препаративный выход. Такое разбиение — один прозрачный алгоритмический шаг, а не универсальный поиск синтеза.

**Перед запуском:** Найдите сложноэфирный фрагмент и примените один заданный шаблон.

In [2]:
from rdkit import Chem
from rdkit.Chem import rdChemReactions
reaction=rdChemReactions.ReactionFromSmarts("[C:1](=[O:2])[O:3][C:4]>>[C:1](=[O:2])[OH:3].[OH][C:4]")
products=reaction.RunReactants((Chem.MolFromSmiles("COC(C)=O"),))
assert products
names=[]
for mol in products[0]:
    Chem.SanitizeMol(mol)
    for atom in mol.GetAtoms(): atom.SetAtomMapNum(0)
    names.append(Chem.MolToSmiles(mol))
print("Учебные предшественники:",names)


Учебные предшественники: ['CC(=O)O', 'CO']


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Ретросинтетический шаблон как гипотеза — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 2. Проверяем продукты

Продукты реакции проходят санитарную проверку RDKit; atom maps убираем только перед представлением SMILES. Сравниваем состав набора предшественников с ожидаемым для учебного эфира. Отсутствие совпадения шаблона означает границу данного правила, не доказанную несинтезируемость цели.

**Перед запуском:** Проверьте, что получены кислота и спирт; это ещё не препаративный протокол.

In [3]:
assert set(names)=={"CC(=O)O","CO"}


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Проверяем продукты — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 3. Сравнение маршрутов в заданной модели

Для последовательных стадий условный общий выход равен произведению долей. Деление assumed_cost на этот выход задаёт только упрощённый сравнительный показатель при согласованной загрузке. Реальная стоимость требует масштаба, стехиометрии, выделения, поставщиков и даты. Здесь цены искусственные; решение сопровождаем списком недостающих сведений.

**Перед запуском:** Сравните два маршрута по произведению долей и заданным суммарным затратам.

In [4]:
routes=pd.read_csv(DATA/"routes.csv")
routes["overall_yield"]=routes.yield1*routes.yield2
routes["normalized_cost"]=routes.assumed_cost/routes.overall_yield
display(routes)
assert np.allclose(routes.overall_yield,[0.56,0.855])
assert routes.loc[1,"normalized_cost"]<routes.loc[0,"normalized_cost"]
routes.to_csv(OUT/"route_comparison.csv",index=False)
artifact={"precursors":names,"routes":routes.to_dict(orient="records"),"scope":"one ester template, not general AI retrosynthesis"}


,route,yield1,yield2,assumed_cost,overall_yield,normalized_cost
0,A,0.80,0.7,600,0.560,1071.428571
1,B,0.95,0.9,750,0.855,877.192982


**После запуска:** сверьте промежуточные переменные и объясните, какое условие проверено. Сравнение маршрутов в заданной модели — часть протокола; успешная ячейка не оценивает самостоятельный ответ.

## 4. Артефакт и граница доказательства

Продукты метилацетата: уксусная кислота и метанол; выходы A=0,56/B=0,855; приведённые затраты A≈1071,43/B≈877,19 при явно заданной модели.

Следующая ячейка сохраняет именно новый запуск и хеши входов. Отсутствующее вычисление нельзя заменять старым output. Время относится к этому компьютеру и не является обещанием производительности.

In [5]:
record={'module':13,'scope':'worked reference execution','python':sys.executable,'elapsed_seconds':time.perf_counter()-started,'result':artifact}
(OUT/'benchmark.json').write_text(json.dumps(record,ensure_ascii=False,indent=2))
print(OUT/'benchmark.json')

/home/miakhlov/course-notebooks/Labs/results/lab13/worked/benchmark.json


## Источники и следующий шаг

1. Исходный вычислительный пример: [Labs/chemistry.py](../chemistry.py), [описание данных](../datasets/README.md).
2. [RDKit: Getting Started](https://www.rdkit.org/docs/GettingStartedInPython.html).
3. [Карта чтения](../../Literature/READING_MAP.md).

Перейдите к [lab13_practice.ipynb](lab13_practice.ipynb). Через неделю повторите первый кейс без просмотра ключа.